# 03 — Built-up Classification via Random Forest

**Goal:** measure whether a trained classifier beats the tuned threshold baseline from notebook
02 (NDBI > 0 AND NDVI < 0.30, **83.6%** agreement with ESA WorldCover), using the same
composite and the same reference dataset.

**Important methodological note:** the threshold baseline was *not* fit to WorldCover — it was
tuned by sweeping a physically-motivated cutoff and picking the peak, so its 83.6% is an honest
out-of-sample number. A Random Forest trained directly on WorldCover-derived labels does not get
that same honesty for free: if evaluated on the same pixels it was trained on, high agreement is
almost guaranteed and proves nothing. So the labeled points are split into training and held-out
test sets up front, and the RF is only ever scored on points it did not train on — the same
standard the baseline was implicitly held to.


In [1]:
import sys
sys.path.insert(0, '../src')

import ee
import geemap
import urllib.request
from acquisition import get_nairobi_boundary, get_sentinel2_composite

ee.Initialize(project='solar-haven-349708')

nairobi = get_nairobi_boundary()
composite, scene_count = get_sentinel2_composite(
    nairobi, start_date='2024-06-01', end_date='2024-09-30', cloud_threshold=20
)
print(f'Composite ready: {scene_count} scenes')

Composite ready: 11 scenes


## Features

Six raw Sentinel-2 reflectance bands plus the two engineered indices from notebook 02 (NDVI,
NDBI) — the indices already proved informative there, so there's no reason to discard that signal
just because the model can now learn its own combinations. Adding them as extra bands lets the
Random Forest use both the raw spectral values and the domain-known ratios.

Bands: B2 (blue), B3 (green), B4 (red), B8 (NIR), B11 (SWIR1), B12 (SWIR2).

In [2]:
bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndbi = composite.normalizedDifference(['B11', 'B8']).rename('NDBI')

feature_image = composite.select(bands).addBands(ndvi).addBands(ndbi)
feature_names = bands + ['NDVI', 'NDBI']
print('Feature bands:', feature_names)

Feature bands: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDBI']


## Labels: ESA WorldCover, sampled and split

Same reference as notebook 02 (`ESA/WorldCover/v200`, class 50 = built-up), but now used as
training/test labels rather than only a post-hoc check. `stratifiedSample` draws a balanced set
of points from each class (built-up vs. not) so the training set isn't dominated by the
majority class — Nairobi is roughly 32% built-up by WorldCover's own count (notebook 02), so an
unstratified sample would be built-up-poor.

A random column splits points 70/30 into train/test *before* anything touches the model, so the
test set stays genuinely held-out.

In [3]:
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').clip(nairobi)
worldcover_builtup = worldcover.eq(50).rename('builtup')

training_image = feature_image.addBands(worldcover_builtup)

samples = training_image.stratifiedSample(
    numPoints=1500,
    classBand='builtup',
    region=nairobi,
    scale=10,
    seed=42,
    geometries=False,
)

samples = samples.randomColumn('split', seed=42)
train_samples = samples.filter(ee.Filter.lt('split', 0.7))
test_samples = samples.filter(ee.Filter.gte('split', 0.7))

print('Train points:', train_samples.size().getInfo())
print('Test points:', test_samples.size().getInfo())

Train points: 2110


Test points: 890


## Train

`ee.Classifier.smileRandomForest` runs server-side in Earth Engine, consistent with the rest of
the pipeline (no need to export pixels to a local array). 100 trees is a standard default —
untuned, same spirit as the baseline's initial threshold-at-0 before it was swept.

In [4]:
classifier = ee.Classifier.smileRandomForest(numberOfTrees=100, seed=42).train(
    features=train_samples,
    classProperty='builtup',
    inputProperties=feature_names,
)

## Evaluate on held-out test points

Accuracy and a confusion matrix, computed only on the 30% split the model never saw during
training — the number to actually compare against the baseline's 83.6%.

In [5]:
test_classified = test_samples.classify(classifier)
confusion = test_classified.errorMatrix('builtup', 'classification')

test_accuracy = confusion.accuracy().getInfo()
print(f'Random Forest held-out test accuracy: {test_accuracy * 100:.1f}%')
print('Confusion matrix (rows=actual, cols=predicted):')
for row in confusion.getInfo():
    print(row)

Random Forest held-out test accuracy: 86.2%
Confusion matrix (rows=actual, cols=predicted):


[375, 74]
[49, 392]


## Classify the full composite and compare to WorldCover

For an apples-to-apples number against notebook 02's methodology (full-image pixel agreement,
not just the point sample), classify every pixel and measure agreement against WorldCover across
all of Nairobi. This number will be somewhat optimistic relative to the held-out test accuracy
above, since WorldCover pixels near training points are spatially autocorrelated with them — the
held-out *point* accuracy above remains the more honest figure, this is included for direct
comparability with the baseline's reported metric.

In [6]:
rf_classified = feature_image.classify(classifier).rename('builtup')

rf_agreement = rf_classified.eq(worldcover_builtup).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=nairobi, scale=10, maxPixels=1e9
).getInfo()['builtup'] * 100

rf_frac = rf_classified.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=nairobi, scale=10, maxPixels=1e9
).getInfo()['builtup'] * 100

print(f'RF full-image agreement with WorldCover: {rf_agreement:.1f}%')
print(f'RF-classified built-up fraction: {rf_frac:.1f}% (WorldCover: 31.9%)')

RF full-image agreement with WorldCover: 86.8%
RF-classified built-up fraction: 38.0% (WorldCover: 31.9%)


## Visualize

In [7]:
builtup_vis = {'min': 0, 'max': 1, 'palette': ['black', 'red']}

Map = geemap.Map(center=[-1.290, 36.868], zoom=11)
Map.addLayer(composite, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'True color', False)
Map.addLayer(rf_classified, builtup_vis, 'Random Forest built-up')
Map.addLayer(worldcover_builtup, builtup_vis, 'WorldCover built-up (reference)', False)
Map.addLayerControl()
Map

Map(center=[-1.29, 36.868], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [8]:
url = rf_classified.getThumbURL({
    'min': 0, 'max': 1, 'palette': ['black', 'red'],
    'region': nairobi, 'dimensions': 800,
})
path = '../data/processed/nairobi_builtup_random_forest.png'
urllib.request.urlretrieve(url, path)
print(f'Saved {path}')

Saved ../data/processed/nairobi_builtup_random_forest.png


## Feature importance

Which bands the model actually leaned on — a sanity check on whether it rediscovered something
like NDBI/NDVI on its own, or found a different signal entirely.

In [9]:
importance = classifier.explain().getInfo()['importance']
for name, score in sorted(importance.items(), key=lambda kv: -kv[1]):
    print(f'{name:6s} {score:.1f}')

B11    393.8
B12    364.6
NDBI   338.0
B3     330.6
NDVI   325.8
B2     318.0
B4     316.6
B8     312.0


## Summary

| Method | Metric | Value |
|---|---|---|
| Threshold baseline (NDBI>0 AND NDVI<0.30) | full-image agreement vs. WorldCover | 83.6% |
| Random Forest | held-out point accuracy (honest comparison) | **86.2%** |
| Random Forest | full-image agreement vs. WorldCover | 86.8% |

**Result: Random Forest wins, 86.2% vs. 83.6%** — a modest but real +2.6 point improvement,
measured on the held-out 30% test split the model never trained on, so it's not an artifact of
evaluating against its own training labels.

Confusion matrix on the held-out set (rows=actual, cols=predicted, order [not-built-up,
built-up]):

```
[375,  74]
[ 49, 392]
```

74 false positives (actual non-built-up predicted as built-up), 49 false negatives — roughly
balanced, unlike the threshold baseline's asymmetric over/under-classification swings seen across
the NDVI sweep in notebook 02.

**Feature importance:** B11 (SWIR1) and B12 (SWIR2) rank highest, with NDBI close behind —
consistent with the domain reasoning that motivated NDBI in the first place. The RF didn't
discover a wildly different signal; it found a smoother, better-calibrated combination of the
same signal already suspected to matter, plus the extra bands (B2/B3/B4/B8) to resolve edge
cases NDBI+NDVI's hard threshold cuts got wrong.

**Trade-off to note:** RF built-up fraction is 38.0% vs. WorldCover's 31.9% — still
over-classifying, just less severely than raw NDBI alone (56.4%) and by more than the tuned
threshold baseline (26.6%). The RF is more *accurate* pixel-by-pixel but not perfectly
*calibrated* in aggregate area — the same distinction flagged in notebook 02 between pixel
agreement and area-total matching.

**Decision: adopt the Random Forest classifier** as the built-up layer going forward — it's the
better-performing option on the metric that matters (held-out accuracy) and its errors are more
balanced than the threshold approach's.